In [2]:
from pydantic import BaseModel, field_serializer
from typing import Optional

In [3]:
from datetime import datetime

class Model(BaseModel):
    dt: Optional[datetime] = None

    @field_serializer("dt", when_used="always")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value

In [4]:
m = Model(dt="2020-01-01T12:00:00")
m

Model(dt=datetime.datetime(2020, 1, 1, 12, 0))

In [5]:
m.model_dump()

type = <class 'datetime.datetime'>


{'dt': datetime.datetime(2020, 1, 1, 12, 0)}

In [6]:
m.model_dump_json()

type = <class 'datetime.datetime'>


'{"dt":"2020-01-01T12:00:00"}'

In [7]:
m = Model()
m


Model(dt=None)

In [8]:
m.model_dump()

type = <class 'NoneType'>


{'dt': None}

In [9]:
m.model_dump_json()

type = <class 'NoneType'>


'{"dt":null}'

In [10]:
from datetime import datetime

class Model(BaseModel):
    dt: Optional[datetime] = None

    @field_serializer("dt", when_used="unless-none")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value

In [11]:
m = Model(dt="2020-01-01T12:00:00")
m

Model(dt=datetime.datetime(2020, 1, 1, 12, 0))

In [12]:
m.model_dump()

type = <class 'datetime.datetime'>


{'dt': datetime.datetime(2020, 1, 1, 12, 0)}

In [13]:
m = Model()
m

Model(dt=None)

In [14]:
m.model_dump()

{'dt': None}

In [15]:
m.model_dump_json()

'{"dt":null}'

In [16]:
from datetime import datetime

class Model(BaseModel):
    dt: Optional[datetime] = None

    @field_serializer("dt", when_used="json-unless-none")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value.strftime("%Y/%-m/%-d %I:%M %p")

In [17]:
m = Model(dt="2020-01-01T12:00:00")
m

Model(dt=datetime.datetime(2020, 1, 1, 12, 0))

In [18]:
m.model_dump_json()

type = <class 'datetime.datetime'>


'{"dt":"2020/1/1 12:00 PM"}'

In [19]:
m.model_dump()

{'dt': datetime.datetime(2020, 1, 1, 12, 0)}

In [20]:
from pydantic import FieldSerializationInfo

In [21]:
class Model(BaseModel):
    dt: Optional[datetime] = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, value, info: FieldSerializationInfo):
        print(f"info={info}")
        return value

In [22]:
m = Model(dt=datetime(2020, 1, 1))
m

Model(dt=datetime.datetime(2020, 1, 1, 0, 0))

In [23]:
m.model_dump()

info=SerializationInfo(include=None, exclude=None, context=None, mode='python', by_alias=False, exclude_unset=False, exclude_defaults=False, exclude_none=False, exclude_computed_fields=False, round_trip=False, serialize_as_any=False)


{'dt': datetime.datetime(2020, 1, 1, 0, 0)}

In [24]:
m.model_dump_json()

info=SerializationInfo(include=None, exclude=None, context=None, mode='json', by_alias=False, exclude_unset=False, exclude_defaults=False, exclude_none=False, exclude_computed_fields=False, round_trip=False, serialize_as_any=False)


'{"dt":"2020-01-01T00:00:00"}'

In [25]:
class Model(BaseModel):
    dt: Optional[datetime] = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, value, info: FieldSerializationInfo):
        print(f"mode_is_json={info.mode_is_json()}")
        return value

In [26]:
m = Model(dt=datetime(2020, 1, 1))

In [27]:
m.model_dump()

mode_is_json=False


{'dt': datetime.datetime(2020, 1, 1, 0, 0)}

In [28]:
m.model_dump_json()

mode_is_json=True


'{"dt":"2020-01-01T00:00:00"}'

In [29]:
import pytz

def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

def dt_utc_json_serializer(dt: datetime) -> str:
    dt = make_utc(dt)
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

In [31]:
class Model(BaseModel):
    dt: Optional[datetime] = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, dt, info: FieldSerializationInfo):
        if info.mode_is_json():
            return dt_utc_json_serializer(dt)
        return make_utc(dt)

In [32]:
m = Model(dt=datetime(2020, 1, 1))
m

Model(dt=datetime.datetime(2020, 1, 1, 0, 0))

In [ ]:
m.model_dump